# NYC Taxi Trip Duration — Model Comparison

Official model: **Ridge(alpha=1)**. This notebook calls shared source code so feature definitions, train-only clustering and preprocessing match the CLI. Test data is never loaded. See `reports/experiment_summary.md` for historical sample results and measured large-data results.

In [ ]:
from pathlib import Path
import sys
from argparse import Namespace
import pandas as pd
from threadpoolctl import threadpool_limits
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.train import run

## Controlled sample exploration

The provided sample split differs from the old 80/20 split of the cleaned EDA export; its scores must not be presented as reproducing the historical sample benchmark. Switch both paths to `split/` for full-data experiments. Every candidate uses the same validation rows. Rejected additions are excluded from later candidates. MLflow is optional.

In [ ]:
args = Namespace(train=ROOT/'split_sample/train.csv', val=ROOT/'split_sample/val.csv',
    model_dir=ROOT/'models/sample', report_dir=ROOT/'reports/sample',
    experiments=True, polynomial=True, benchmarks=False, n_clusters=5, jobs=4, mlflow_uri=None)
with threadpool_limits(limits=args.jobs):
    metadata = run(args)

In [ ]:
pd.read_csv(args.report_dir/'feature_ablation.csv')

In [ ]:
pd.read_csv(args.report_dir/'benchmark_results.csv')

## Interpretation

Compare changes against the retained incumbent and original baseline. An RMSE decrease greater than 0.000001 is retained; this numerical rule does not establish statistical significance. Polynomial and tree models are diagnostic benchmarks and cannot replace the official plain Ridge. All reported metrics use `log1p(trip_duration)`.